<a href="https://colab.research.google.com/github/roughhawkbit/digi-inno-road-prod/blob/main/analysis/2_0_random_sample_firm_responses.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook randomly samples firms which have at least some text for the key questions, then prepares the text for all firms in a format ready to go into a Google Doc. This document will then be used as part of a manual assessment of the WCC of the sample firms.

# Setup

In [ ]:
import os
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    repo_path = '/content/drive/MyDrive/digi-inno-road-prod'
    if os.path.isdir(repo_path):
      cwd = os.getcwd()
      os.chdir(repo_path)
      !git pull
      os.chdir(cwd)
    else:
      !git clone https://github.com/roughhawkbit/digi-inno-road-prod.git /content/drive/MyDrive/digi-inno-road-prod
      print('Repository cloned into your Google Drive. It is strongly recommended that you copy the credentials.json, sheet.json, and token.json files into the secrets folder before proceeding.')
    sys.path.insert(0, repo_path)
    IN_COLAB = True
except ImportError:
    repo_path = os.path.abspath(os.path.join('../src'))
    IN_COLAB = False

if not repo_path in sys.path:
    sys.path.insert(0, repo_path)

In [ ]:
if IN_COLAB:
  output_path = os.path.join(repo_path, 'analysis', 'outputs')
else:
  output_path = os.path.join('.', 'outputs')
output_path = os.path.abspath(output_path)

# Import packages & data

Conda packages

In [ ]:
import numpy
import pandas

Project sourcecode packages

In [ ]:
from innoprod.sheet_tools import get_sheet_dfs
from innoprod.wrangling.msyh_data_sharing import wrangle_roadmaps
from innoprod.wrangling.wrangling_tools import is_non_empty

Data

In [ ]:
data = get_sheet_dfs()

INCLUDE_NO_GRANTS_FIRMS = True
if INCLUDE_NO_GRANTS_FIRMS:
  roadmaps_df = pandas.concat([data['Roadmaps'], data['RoadmapsWithoutGrants']])
else:
  roadmaps_df = data['Roadmaps']
roadmaps_df = wrangle_roadmaps(roadmaps_df)

# Prepare & quantify text

In [ ]:
key_questions = [
    'Summary review of Edge Digital diagnostic report & current state and key improvement areas',
    'What are the internal barriers to growth? How do you intend to finance future growth? Are there sufficient leadership and management skills in the business to achieve your growth? What opportunities do you have to expand into new markets?',
    'Details of any existing Digital Strategy',
    'Level of current Strategic Digital Skills/knowledge in the business',
    'Level of current Technical Digital Skills/knowledge in the business',
    'Whether the business is already investing/adopting/utilising Industry 4.0 Technologies, with examples',
    'Summary of the identified problems, including Gap Analysis'
]

In [ ]:
roadmaps_df = roadmaps_df[['Client ID']+key_questions].fillna('')
# roadmaps_df

In [ ]:
roadmaps_df[roadmaps_df[key_questions] == 'nan'] = ''
# roadmaps_df

In [ ]:
roadmaps_df['Word Count'] = roadmaps_df.apply(lambda row: sum([len(str(row[q]).split()) for q in key_questions]), axis=1)
# roadmaps_df

In [ ]:
roadmaps_df = roadmaps_df[roadmaps_df['Word Count'] > 0].drop(columns=['Word Count']).reset_index(drop=True)
# roadmaps_df

# Select random sample

In [ ]:
random_seed = 42
numpy.random.seed(random_seed)

In [ ]:
n_clients = 30

In [ ]:
sample_df = roadmaps_df.sample(n_clients).reset_index(drop=True)
sample_df.index += 1
sample_df

In [ ]:
for index, row in sample_df.iterrows():
  print(f'Firm {index}')
  print(f'ID: {row["Client ID"]}')
  for q in key_questions:
    print(f'\n{q}')
    print(row[q])
  print('\n------------------------------------------')

# Write to file

TODO: the code below does not yet work (problem defining scopes). The workaround is to copy the text output above, paste into a Google Doc, and manually add formatting.

In [ ]:
os.listdir('/content/drive/MyDrive/Sample Firm Texts/')

In [ ]:
from innoprod.path_tools import secrets_path
from google.oauth2.credentials import Credentials

SCOPES = ["https://www.googleapis.com/auth/documents"]
token_path = os.path.abspath(os.path.join(secrets_path(), "token.json"))
credentials = Credentials.from_authorized_user_file(token_path, SCOPES)

# credentials = get_credentials()
docs_service = build('docs', 'v1', credentials=credentials)
# drive_service = build('drive', 'v3', credentials=credentials)

# TODO: automate creation of this folder and retrieval of the ID
# target_folder_id = '1xzpN6RGiSok2QlaQ9i3rXwdnymSgffXz'

In [ ]:
new_doc_id = '1go8a_2zUBtVef89g3Q4t_wQdhwo1w6dhlUrR82qBpwE'


In [ ]:
all_text = ''.join(sample_df['Concatenated Text'])
all_text

In [ ]:
# for index, row in sample_df.iterrows():
  # doc_metadata = {'title': f'Sample text for firm {row["Client ID"]}'}
  # Create an empty document and get its ID
  # new_doc = docs_service.documents().create(body=doc_metadata).execute()
  # new_doc_id = new_doc.get('documentId')
  # Insert the text into the new document
requests = [
    {'insertText': {
            'location': {'index': 1},
            # 'text': row['Concatenated text']
            'text': all_text
    }}
]
docs_service.documents().batchUpdate(
  documentId=new_doc_id,
  body={'requests': requests}
).execute()
  # Move the document into the target folder
  # file_info = drive_service.files().get(
  #       fileId=new_doc_id,
  #       fields='parents'
  #   ).execute()
  # previous_parents = ",".join(file_info.get('parents', []))
  # moved_file = drive_service.files().update(
  #       fileId=new_doc_id,
  #       addParents=target_folder_id,
  #       removeParents=previous_parents,
  #       fields='id, parents'
  #   ).execute()